In [2]:
!pip install pandas tqdm pyarrow requests -q

In [3]:
import pandas as pd
import requests, zipfile, io, os
from datetime import datetime, timedelta
from tqdm import tqdm

In [4]:
# -----------------------------
# GDELT 2.0 Event column names
# -----------------------------

GDELT_COLUMNS = [
    "GLOBALEVENTID", "SQLDATE", "MonthYear", "Year", "FractionDate",
    "Actor1Code", "Actor1Name", "Actor1CountryCode", "Actor1KnownGroupCode",
    "Actor1EthnicCode", "Actor1Religion1Code", "Actor1Religion2Code",
    "Actor1Type1Code", "Actor1Type2Code", "Actor1Type3Code",
    "Actor2Code", "Actor2Name", "Actor2CountryCode", "Actor2KnownGroupCode",
    "Actor2EthnicCode", "Actor2Religion1Code", "Actor2Religion2Code",
    "Actor2Type1Code", "Actor2Type2Code", "Actor2Type3Code",
    "IsRootEvent", "EventCode", "EventBaseCode", "EventRootCode",
    "QuadClass", "GoldsteinScale", "NumMentions", "NumSources",
    "NumArticles", "AvgTone",
    "Actor1Geo_Type", "Actor1Geo_FullName", "Actor1Geo_CountryCode",
    "Actor1Geo_ADM1Code", "Actor1Geo_ADM2Code", "Actor1Geo_Lat",
    "Actor1Geo_Long", "Actor1Geo_FeatureID",
    "Actor2Geo_Type", "Actor2Geo_FullName", "Actor2Geo_CountryCode",
    "Actor2Geo_ADM1Code", "Actor2Geo_ADM2Code", "Actor2Geo_Lat",
    "Actor2Geo_Long", "Actor2Geo_FeatureID",
    "ActionGeo_Type", "ActionGeo_FullName", "ActionGeo_CountryCode",
    "ActionGeo_ADM1Code", "ActionGeo_ADM2Code", "ActionGeo_Lat",
    "ActionGeo_Long", "ActionGeo_FeatureID",
    "DATEADDED", "SOURCEURL"
]

In [5]:
# --------------------------------------------------
# Critical event taxonomy for trade-risk analysis
# --------------------------------------------------

CRITICAL_EVENT_ROOTS = {
    "14": "protest",
    "17": "coercion",
    "18": "assault",
    "19": "fight",
    "20": "mass_violence"
}

# Optional stricter CAMEO event-code filters
# These are examples; you can expand later.
CRITICAL_EVENT_CODES = {
    # Protests / unrest
    "141": "demonstrate",
    "142": "conduct_hunger_strike",
    "143": "conduct_strike_or_boycott",
    "144": "obstruct_passage",
    "145": "protest_violently",

    # Coercion
    "171": "seize_or_damage_property",
    "172": "impose_administrative_sanctions",
    "173": "arrest_detain",
    "174": "expel",
    "175": "use_repression",

    # Assault
    "181": "abduct",
    "182": "physically_assault",
    "183": "conduct_suicide_attack",
    "184": "use_as_human_shield",
    "185": "attempt_assassination",
    "186": "assassinate",

    # Fight
    "190": "use_conventional_military_force",
    "191": "impose_blockade",
    "192": "occupy_territory",
    "193": "fight_with_small_arms",
    "194": "fight_with_artillery",
    "195": "employ_aerial_weapons",

    # Mass violence
    "200": "use_unconventional_mass_violence",
    "201": "engage_in_mass_expulsion",
    "202": "engage_in_mass_killings",
    "203": "engage_in_ethnic_cleansing",
    "204": "use_weapons_of_mass_destruction"
}

In [6]:
def generate_gdelt_timestamps(start_date, end_date):
    """
    Generate GDELT 2.0 15-minute timestamps between two dates.
    Dates should be strings: 'YYYY-MM-DD'.
    """
    start = datetime.strptime(start_date, "%Y-%m-%d")
    end = datetime.strptime(end_date, "%Y-%m-%d") + timedelta(days=1)

    timestamps = []
    current = start

    while current < end:
        timestamps.append(current.strftime("%Y%m%d%H%M%S"))
        current += timedelta(minutes=15)

    return timestamps

In [7]:
def download_gdelt_file(timestamp):
    """
    Download one GDELT 2.0 event file.
    """
    url = f"http://data.gdeltproject.org/gdeltv2/{timestamp}.export.CSV.zip"

    try:
        r = requests.get(url, timeout=20)

        if r.status_code != 200:
            return None

        z = zipfile.ZipFile(io.BytesIO(r.content))
        csv_name = z.namelist()[0]

        df = pd.read_csv(
            z.open(csv_name),
            sep="\t",
            header=None,
            names=GDELT_COLUMNS,
            dtype=str,
            low_memory=False
        )

        return df

    except Exception:
        return None

In [8]:
def filter_critical_events(
    df,
    countries=None,
    min_mentions=5,
    min_sources=2,
    use_strict_codes=False
):
    """
    Filter GDELT events for critical trade-relevant events.

    countries:
        list of ISO-like GDELT country codes, e.g. ['UKR', 'RUS', 'CHN']

    use_strict_codes:
        False = keep broad root codes 14,17,18,19,20
        True  = keep only selected critical EventCode values
    """

    df = df.copy()

    df["NumMentions"] = pd.to_numeric(df["NumMentions"], errors="coerce")
    df["NumSources"] = pd.to_numeric(df["NumSources"], errors="coerce")
    df["GoldsteinScale"] = pd.to_numeric(df["GoldsteinScale"], errors="coerce")
    df["AvgTone"] = pd.to_numeric(df["AvgTone"], errors="coerce")

    if use_strict_codes:
        df = df[df["EventCode"].isin(CRITICAL_EVENT_CODES.keys())]
        df["event_class"] = df["EventCode"].map(CRITICAL_EVENT_CODES)
    else:
        df = df[df["EventRootCode"].isin(CRITICAL_EVENT_ROOTS.keys())]
        df["event_class"] = df["EventRootCode"].map(CRITICAL_EVENT_ROOTS)

    df = df[
        (df["NumMentions"] >= min_mentions) &
        (df["NumSources"] >= min_sources)
    ]

    if countries is not None:
        countries = set(countries)
        df = df[
            df["Actor1CountryCode"].isin(countries) |
            df["Actor2CountryCode"].isin(countries) |
            df["ActionGeo_CountryCode"].isin(countries)
        ]

    keep_cols = [
        "GLOBALEVENTID",
        "SQLDATE",
        "DATEADDED",
        "Actor1Name",
        "Actor1CountryCode",
        "Actor2Name",
        "Actor2CountryCode",
        "ActionGeo_CountryCode",
        "ActionGeo_FullName",
        "EventCode",
        "EventBaseCode",
        "EventRootCode",
        "event_class",
        "QuadClass",
        "GoldsteinScale",
        "NumMentions",
        "NumSources",
        "NumArticles",
        "AvgTone",
        "SOURCEURL"
    ]

    return df[keep_cols]

In [9]:
def extract_gdelt_critical_events(
    start_date,
    end_date,
    countries=None,
    min_mentions=5,
    min_sources=2,
    use_strict_codes=False,
    output_path="gdelt_critical_events.parquet"
):
    """
    Main extraction function.
    """

    timestamps = generate_gdelt_timestamps(start_date, end_date)
    all_events = []

    for ts in tqdm(timestamps):
        df = download_gdelt_file(ts)

        if df is None or df.empty:
            continue

        filtered = filter_critical_events(
            df,
            countries=countries,
            min_mentions=min_mentions,
            min_sources=min_sources,
            use_strict_codes=use_strict_codes
        )

        if not filtered.empty:
            all_events.append(filtered)

    if len(all_events) == 0:
        print("No matching events found.")
        return pd.DataFrame()

    result = pd.concat(all_events, ignore_index=True)

    result["date"] = pd.to_datetime(result["SQLDATE"], format="%Y%m%d", errors="coerce")
    result = result.drop_duplicates(subset=["GLOBALEVENTID"])

    result.to_parquet(output_path, index=False)

    print(f"Saved {len(result):,} events to {output_path}")

    return result

In [10]:
events = extract_gdelt_critical_events(
    start_date="2024-01-01",
    end_date="2024-01-07",
    countries=["UKR", "RUS"],
    min_mentions=10,
    min_sources=3,
    use_strict_codes=False,
    output_path="ukraine_russia_critical_events.parquet"
)

events.head()

100%|██████████| 672/672 [05:19<00:00,  2.10it/s]

Saved 36 events to ukraine_russia_critical_events.parquet


,GLOBALEVENTID,SQLDATE,DATEADDED,Actor1Name,Actor1CountryCode,Actor2Name,Actor2CountryCode,ActionGeo_CountryCode,ActionGeo_FullName,EventCode,...,EventRootCode,event_class,QuadClass,GoldsteinScale,NumMentions,NumSources,NumArticles,AvgTone,SOURCEURL,date
0,1149158665,20240101,20240101000000,UKRAINIAN,UKR,BELGOROD,RUS,RS,"Belgorod, Belgorodskaya Oblast', Russia",190,...,19,fight,4,-10.0,18,3,18,-8.391897,https://www.watoday.com.au/world/europe/in-esc...,2024-01-01
1,1149191247,20240101,20240101081500,RUSSIAN,RUS,UKRAINIAN,UKR,RS,Russia,193,...,19,fight,4,-10.0,22,3,22,-5.840241,https://www.easternriverinachronicle.com.au/st...,2024-01-01
2,1149325529,20240102,20240102114500,UKRAINIAN,UKR,RUSSIA,RUS,RS,"Belgorod, Belgorodskaya Oblast', Russia",194,...,19,fight,4,-10.0,18,3,18,-8.200008,https://www.thetelegraph.com/news/world/articl...,2024-01-02
3,1149363223,20240102,20240102163000,MILITARY,NaN,UKRAINE,UKR,UP,"Kyiv, Kyyiv, Misto, Ukraine",193,...,19,fight,4,-10.0,16,4,16,-4.728132,https://www.wcsufm.org/latest-from-npr/2024-01...,2024-01-02
4,1149363225,20240102,20240102163000,MILITARY,NaN,UKRAINIAN,UKR,UP,"Kyiv, Kyyiv, Misto, Ukraine",193,...,19,fight,4,-10.0,16,4,16,-4.728132,https://www.wcsufm.org/latest-from-npr/2024-01...,2024-01-02


In [11]:
!pip install pandas tqdm pyarrow requests -q

In [12]:
import pandas as pd
import requests, zipfile, io
from datetime import datetime, timedelta
from tqdm import tqdm

In [13]:
GDELT_COLUMNS = [
    "GLOBALEVENTID", "SQLDATE", "MonthYear", "Year", "FractionDate",
    "Actor1Code", "Actor1Name", "Actor1CountryCode", "Actor1KnownGroupCode",
    "Actor1EthnicCode", "Actor1Religion1Code", "Actor1Religion2Code",
    "Actor1Type1Code", "Actor1Type2Code", "Actor1Type3Code",
    "Actor2Code", "Actor2Name", "Actor2CountryCode", "Actor2KnownGroupCode",
    "Actor2EthnicCode", "Actor2Religion1Code", "Actor2Religion2Code",
    "Actor2Type1Code", "Actor2Type2Code", "Actor2Type3Code",
    "IsRootEvent", "EventCode", "EventBaseCode", "EventRootCode",
    "QuadClass", "GoldsteinScale", "NumMentions", "NumSources",
    "NumArticles", "AvgTone",
    "Actor1Geo_Type", "Actor1Geo_FullName", "Actor1Geo_CountryCode",
    "Actor1Geo_ADM1Code", "Actor1Geo_ADM2Code", "Actor1Geo_Lat",
    "Actor1Geo_Long", "Actor1Geo_FeatureID",
    "Actor2Geo_Type", "Actor2Geo_FullName", "Actor2Geo_CountryCode",
    "Actor2Geo_ADM1Code", "Actor2Geo_ADM2Code", "Actor2Geo_Lat",
    "Actor2Geo_Long", "Actor2Geo_FeatureID",
    "ActionGeo_Type", "ActionGeo_FullName", "ActionGeo_CountryCode",
    "ActionGeo_ADM1Code", "ActionGeo_ADM2Code", "ActionGeo_Lat",
    "ActionGeo_Long", "ActionGeo_FeatureID",
    "DATEADDED", "SOURCEURL"
]

In [14]:
SEMICONDUCTOR_COUNTRIES = [
    "TWN",  # Taiwan
    "CHN",  # China
    "KOR",  # South Korea
    "JPN",  # Japan
    "USA",  # United States
    "NLD",  # Netherlands / ASML
    "DEU",  # Germany
    "SGP",  # Singapore
    "MYS",  # Malaysia
    "PHL",  # Philippines
    "VNM",  # Vietnam
    "THA",  # Thailand
    "IND",  # India
    "ISR"   # Israel
]

In [15]:
SEMICONDUCTOR_KEYWORDS = [
    # chips / fabs
    "semiconductor", "semiconductors", "chip", "chips", "microchip",
    "foundry", "fab", "wafer", "silicon wafer", "integrated circuit",

    # firms
    "tsmc", "samsung electronics", "sk hynix", "intel", "micron",
    "asml", "tokyo electron", "applied materials", "lam research",
    "globalfoundries", "smic", "umc",

    # export controls / sanctions
    "export control", "export controls", "sanction", "sanctions",
    "blacklist", "entity list", "technology ban", "chip ban",
    "trade restriction", "licensing restriction",

    # inputs
    "neon", "argon", "krypton", "xenon", "photoresist",
    "palladium", "gallium", "germanium", "rare earth",

    # supply chain / logistics
    "port closure", "shipping disruption", "supply chain",
    "factory shutdown", "power outage", "electricity shortage",

    # geopolitical risk
    "taiwan strait", "blockade", "military drill", "missile",
    "invasion", "war", "conflict"
]

In [16]:
SEMICONDUCTOR_EVENT_ROOTS = {
    "14": "civil_unrest",
    "17": "coercion_sanctions_controls",
    "18": "assault_security_risk",
    "19": "military_conflict",
    "20": "mass_violence"
}

SEMICONDUCTOR_EVENT_CODES = {
    # Trade / sanctions / coercion
    "172": "sanctions_or_admin_restrictions",
    "173": "arrest_or_detain",
    "174": "expel_or_deport",
    "175": "repression",

    # Protests / strikes
    "143": "strike_or_boycott",
    "144": "obstruct_passage",
    "145": "violent_protest",

    # Military / blockade / conflict
    "190": "military_force",
    "191": "blockade",
    "192": "occupation",
    "193": "small_arms_conflict",
    "194": "artillery_conflict",
    "195": "aerial_weapons",

    # Severe violence
    "200": "mass_violence"
}

In [17]:
def generate_gdelt_timestamps(start_date, end_date):
    start = datetime.strptime(start_date, "%Y-%m-%d")
    end = datetime.strptime(end_date, "%Y-%m-%d") + timedelta(days=1)

    timestamps = []
    current = start

    while current < end:
        timestamps.append(current.strftime("%Y%m%d%H%M%S"))
        current += timedelta(minutes=15)

    return timestamps

In [18]:
def download_gdelt_file(timestamp):
    url = f"http://data.gdeltproject.org/gdeltv2/{timestamp}.export.CSV.zip"

    try:
        r = requests.get(url, timeout=20)

        if r.status_code != 200:
            return None

        z = zipfile.ZipFile(io.BytesIO(r.content))
        csv_name = z.namelist()[0]

        return pd.read_csv(
            z.open(csv_name),
            sep="\t",
            header=None,
            names=GDELT_COLUMNS,
            dtype=str,
            low_memory=False
        )

    except Exception:
        return None

In [19]:
def contains_semiconductor_keyword(row):
    text = " ".join([
        str(row.get("Actor1Name", "")),
        str(row.get("Actor2Name", "")),
        str(row.get("ActionGeo_FullName", "")),
        str(row.get("SOURCEURL", ""))
    ]).lower()

    return any(keyword in text for keyword in SEMICONDUCTOR_KEYWORDS)

In [20]:
def classify_semiconductor_event(row):
    event_code = str(row["EventCode"])
    root_code = str(row["EventRootCode"])

    if event_code in SEMICONDUCTOR_EVENT_CODES:
        return SEMICONDUCTOR_EVENT_CODES[event_code]

    if root_code in SEMICONDUCTOR_EVENT_ROOTS:
        return SEMICONDUCTOR_EVENT_ROOTS[root_code]

    return "other_semiconductor_relevant"

In [21]:
def filter_semiconductor_supply_chain_events(
    df,
    countries=SEMICONDUCTOR_COUNTRIES,
    min_mentions=5,
    min_sources=2,
    require_keyword=True
):
    df = df.copy()

    for col in ["NumMentions", "NumSources", "NumArticles", "GoldsteinScale", "AvgTone"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df[
        df["Actor1CountryCode"].isin(countries) |
        df["Actor2CountryCode"].isin(countries) |
        df["ActionGeo_CountryCode"].isin(countries)
    ]

    if df.empty:
        return pd.DataFrame()

    df = df[
        df["EventRootCode"].isin(SEMICONDUCTOR_EVENT_ROOTS.keys()) |
        df["EventCode"].isin(SEMICONDUCTOR_EVENT_CODES.keys())
    ]

    if df.empty:
        return pd.DataFrame()

    df = df[
        (df["NumMentions"] >= min_mentions) &
        (df["NumSources"] >= min_sources)
    ]

    if df.empty:
        return pd.DataFrame()

    if require_keyword:
        df = df[df.apply(contains_semiconductor_keyword, axis=1)]

    if df.empty:
        return pd.DataFrame()

    df["event_class"] = df.apply(classify_semiconductor_event, axis=1)

    keep_cols = [
        "GLOBALEVENTID",
        "SQLDATE",
        "DATEADDED",
        "Actor1Name",
        "Actor1CountryCode",
        "Actor2Name",
        "Actor2CountryCode",
        "ActionGeo_CountryCode",
        "ActionGeo_FullName",
        "EventCode",
        "EventBaseCode",
        "EventRootCode",
        "event_class",
        "GoldsteinScale",
        "NumMentions",
        "NumSources",
        "NumArticles",
        "AvgTone",
        "SOURCEURL"
    ]

    return df[keep_cols]

In [22]:
def extract_semiconductor_gdelt_events(
    start_date,
    end_date,
    countries=SEMICONDUCTOR_COUNTRIES,
    min_mentions=5,
    min_sources=2,
    require_keyword=True,
    output_path="gdelt_semiconductor_supply_chain_events.parquet"
):
    timestamps = generate_gdelt_timestamps(start_date, end_date)
    all_events = []

    for ts in tqdm(timestamps):
        df = download_gdelt_file(ts)

        if df is None or df.empty:
            continue

        filtered = filter_semiconductor_supply_chain_events(
            df=df,
            countries=countries,
            min_mentions=min_mentions,
            min_sources=min_sources,
            require_keyword=require_keyword
        )

        if not filtered.empty:
            all_events.append(filtered)

    if not all_events:
        print("No matching semiconductor supply-chain events found.")
        return pd.DataFrame()

    result = pd.concat(all_events, ignore_index=True)

    result["date"] = pd.to_datetime(result["SQLDATE"], format="%Y%m%d", errors="coerce")
    result = result.drop_duplicates(subset=["GLOBALEVENTID"])

    result.to_parquet(output_path, index=False)

    print(f"Saved {len(result):,} events to {output_path}")

    return result

In [23]:
events = extract_semiconductor_gdelt_events(
    start_date="2024-01-01",
    end_date="2024-01-31",
    countries=SEMICONDUCTOR_COUNTRIES,
    min_mentions=5,
    min_sources=2,
    require_keyword=True,
    output_path="gdelt_semiconductor_events.parquet"
)

events.head()

100%|██████████| 2976/2976 [24:13<00:00,  2.05it/s]

Saved 796 events to gdelt_semiconductor_events.parquet


,GLOBALEVENTID,SQLDATE,DATEADDED,Actor1Name,Actor1CountryCode,Actor2Name,Actor2CountryCode,ActionGeo_CountryCode,ActionGeo_FullName,EventCode,EventBaseCode,EventRootCode,event_class,GoldsteinScale,NumMentions,NumSources,NumArticles,AvgTone,SOURCEURL,date
0,1149157629,20240101,20240101000000,NaN,NaN,ISRAEL,ISR,IS,"Gaza, Israel (general), Israel",141,141,14,civil_unrest,-6.5,8,2,8,-6.896552,https://www.michiganradio.org/2023-12-31/israe...,2024-01-01
1,1149157960,20240101,20240101000000,GERMAN,DEU,ISLAMIC,NaN,GM,"Berlin, Berlin, Germany",173,173,17,arrest_or_detain,-5.0,6,3,6,-2.099453,https://www.thedailystar.com/news/national/new...,2024-01-01
2,1149157961,20240101,20240101000000,GERMAN,DEU,ISLAMIC,NaN,US,"Times Square, New York, United States",173,173,17,arrest_or_detain,-5.0,6,3,6,-2.099453,https://www.thedailystar.com/news/national/new...,2024-01-01
3,1149157969,20240101,20240101000000,GERMAN,DEU,COLOGNE,DEU,GM,"Berlin, Berlin, Germany",190,190,19,military_force,-10.0,18,3,18,-2.099453,https://www.thedailystar.com/news/national/new...,2024-01-01
4,1149157972,20240101,20240101000000,GERMAN,DEU,ISLAMIC,NaN,GM,"Berlin, Berlin, Germany",173,173,17,arrest_or_detain,-5.0,18,3,18,-2.099453,https://www.thedailystar.com/news/national/new...,2024-01-01


In [24]:
events["month"] = events["date"].dt.to_period("M").astype(str)

monthly_semiconductor_risk = (
    events
    .groupby(["month", "ActionGeo_CountryCode", "event_class"])
    .agg(
        event_count=("GLOBALEVENTID", "count"),
        avg_goldstein=("GoldsteinScale", "mean"),
        avg_tone=("AvgTone", "mean"),
        total_mentions=("NumMentions", "sum"),
        total_sources=("NumSources", "sum"),
        total_articles=("NumArticles", "sum")
    )
    .reset_index()
)

monthly_semiconductor_risk.head()

,month,ActionGeo_CountryCode,event_class,event_count,avg_goldstein,avg_tone,total_mentions,total_sources,total_articles
0,2023-01,IS,arrest_or_detain,2,-5.0,-6.943740,24,6,24
1,2023-01,IS,assault_security_risk,1,-9.0,-6.753813,52,13,52
2,2023-01,US,arrest_or_detain,1,-5.0,-4.186930,10,5,10
3,2023-01,US,sanctions_or_admin_restrictions,1,-5.0,-6.625072,12,2,12
4,2023-12,DJ,aerial_weapons,3,-10.0,-5.883544,92,22,92


In [25]:
monthly_semiconductor_risk.to_csv(
    "monthly_semiconductor_supply_chain_risk.csv",
    index=False
)

In [26]:
import os
os.getcwd()

'/home/hefouzinho/electronic_prods_pred'

In [27]:
#import os

#os.listdir("/content")

In [28]:
#from google.colab import files

#files.download("/content/monthly_semiconductor_supply_chain_risk.csv")

In [29]:
import os, pandas as pd
from datetime import datetime, timedelta


In [30]:
# =====================================================================
# GDELT 1.0 DAILY downloader — FAST route (~2,557 files vs ~245,000)
# Reuses filter_semiconductor_supply_chain_events etc. unchanged;
# only the column layout + URL differ from your 2.0 code.
# =====================================================================
import os
CHECKPOINT_DIR = "gdelt_chunks"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# GDELT 1.0 = 58 columns (no ADM2 fields; DATEADDED is YYYYMMDD)
GDELT_COLUMNS_V1 = [
    "GLOBALEVENTID","SQLDATE","MonthYear","Year","FractionDate",
    "Actor1Code","Actor1Name","Actor1CountryCode","Actor1KnownGroupCode",
    "Actor1EthnicCode","Actor1Religion1Code","Actor1Religion2Code",
    "Actor1Type1Code","Actor1Type2Code","Actor1Type3Code",
    "Actor2Code","Actor2Name","Actor2CountryCode","Actor2KnownGroupCode",
    "Actor2EthnicCode","Actor2Religion1Code","Actor2Religion2Code",
    "Actor2Type1Code","Actor2Type2Code","Actor2Type3Code",
    "IsRootEvent","EventCode","EventBaseCode","EventRootCode","QuadClass",
    "GoldsteinScale","NumMentions","NumSources","NumArticles","AvgTone",
    "Actor1Geo_Type","Actor1Geo_FullName","Actor1Geo_CountryCode",
    "Actor1Geo_ADM1Code","Actor1Geo_Lat","Actor1Geo_Long","Actor1Geo_FeatureID",
    "Actor2Geo_Type","Actor2Geo_FullName","Actor2Geo_CountryCode",
    "Actor2Geo_ADM1Code","Actor2Geo_Lat","Actor2Geo_Long","Actor2Geo_FeatureID",
    "ActionGeo_Type","ActionGeo_FullName","ActionGeo_CountryCode",
    "ActionGeo_ADM1Code","ActionGeo_Lat","ActionGeo_Long","ActionGeo_FeatureID",
    "DATEADDED","SOURCEURL",
]

def generate_gdelt_dates(start_date, end_date):
    cur = datetime.strptime(start_date, "%Y-%m-%d")
    end = datetime.strptime(end_date, "%Y-%m-%d")
    days = []
    while cur <= end:
        days.append(cur.strftime("%Y%m%d")); cur += timedelta(days=1)
    return days

def download_gdelt_file_v1(date_str):
    """Download ONE GDELT 1.0 daily file (date_str = 'YYYYMMDD')."""
    url = f"http://data.gdeltproject.org/events/{date_str}.export.CSV.zip"
    try:
        r = requests.get(url, timeout=60)
        if r.status_code != 200:
            return None
        z = zipfile.ZipFile(io.BytesIO(r.content))
        raw = pd.read_csv(z.open(z.namelist()[0]), sep="\t",
                          header=None, dtype=str, low_memory=False)
        if raw.shape[1] != len(GDELT_COLUMNS_V1):
            print(f"  ⚠ {date_str}: {raw.shape[1]} cols (expected "
                  f"{len(GDELT_COLUMNS_V1)}) — schema mismatch")
            return None
        raw.columns = GDELT_COLUMNS_V1
        return raw
    except Exception:
        return None

def extract_semiconductor_gdelt_events_v1(
    start_date, end_date,
    countries=SEMICONDUCTOR_COUNTRIES,
    min_mentions=5, min_sources=2, require_keyword=False,
    output_path="gdelt_semiconductor_events_v1.parquet",
):
    all_events = []
    for d in tqdm(generate_gdelt_dates(start_date, end_date)):
        df = download_gdelt_file_v1(d)
        if df is None or df.empty:
            continue
        filt = filter_semiconductor_supply_chain_events(
            df, countries=countries, min_mentions=min_mentions,
            min_sources=min_sources, require_keyword=require_keyword)
        if not filt.empty:
            all_events.append(filt)
    if not all_events:
        print("No matching events found."); return pd.DataFrame()
    result = pd.concat(all_events, ignore_index=True)
    result["date"] = pd.to_datetime(result["SQLDATE"], format="%Y%m%d", errors="coerce")
    result = result.drop_duplicates(subset=["GLOBALEVENTID"])
    result.to_parquet(output_path, index=False)
    print(f"Saved {len(result):,} events to {output_path}")
    return result

def _month_ranges(y0, y1):
    out, cur, end = [], datetime(y0,1,1), datetime(y1,12,31)
    while cur <= end:
        nxt = datetime(cur.year + (cur.month==12), (cur.month % 12)+1, 1)
        out.append((cur.strftime("%Y-%m-%d"),
                    (nxt-timedelta(days=1)).strftime("%Y-%m-%d")))
        cur = nxt
    return out

def pull_gdelt_years_v1(y0=2017, y1=2023, require_keyword=False):
    """Checkpointed, resumable 1.0-daily pull. Safe to stop & re-run."""
    chunk_files = []
    for start, end in _month_ranges(y0, y1):
        tag = start[:7]
        out = f"{CHECKPOINT_DIR}/gdelt_chip_v1_{tag}.parquet"
        chunk_files.append(out)
        if os.path.exists(out):
            print(f"✓ {tag} done — skipping"); continue
        print(f"\n=== {tag} (1.0 daily) ===")
        extract_semiconductor_gdelt_events_v1(
            start, end, require_keyword=require_keyword, output_path=out)
    parts = [pd.read_parquet(f) for f in chunk_files if os.path.exists(f)]
    events = pd.concat(parts, ignore_index=True).drop_duplicates("GLOBALEVENTID")
    events.to_parquet("gdelt_chip_events_2017_2023.parquet", index=False)
    print(f"\nFINAL: {len(events):,} events -> gdelt_chip_events_2017_2023.parquet")
    return events

In [31]:
test = download_gdelt_file_v1("20220103")
if test is None:
    print("No file / schema mismatch — tell me and we fall back to 2.0.")
else:
    print("shape:", test.shape)
    print(test[["GLOBALEVENTID","SQLDATE","Actor1CountryCode","Actor2CountryCode",
                "EventRootCode","GoldsteinScale","NumMentions","SOURCEURL"]].head(3).to_string())

shape: (78231, 58)
  GLOBALEVENTID   SQLDATE Actor1CountryCode Actor2CountryCode EventRootCode GoldsteinScale NumMentions                                                                                                                          SOURCEURL
0    1021574062  20210103               NaN               TWN            01            0.0          10  http://www.msn.com/en-us/news/world/stop-counting-warships-china-s-special-operations-forces-are-taiwan-s-real-problem/ar-AASmMx7
1    1021574063  20210103               NaN               NaN            01            0.0          10                                                      https://www.chronicle.co.zw/comment-armed-robbers-should-not-taint-our-peace/
2    1021574064  20210103               NaN               USA            02            3.0           6                                http://www.bleedingheartland.com/2021/01/01/best-of-bleeding-heartlands-original-reporting-in-2020/


In [32]:
events = pull_gdelt_years_v1(2017, 2023)

✓ 2017-01 done — skipping
✓ 2017-02 done — skipping
✓ 2017-03 done — skipping
✓ 2017-04 done — skipping
✓ 2017-05 done — skipping
✓ 2017-06 done — skipping
✓ 2017-07 done — skipping
✓ 2017-08 done — skipping
✓ 2017-09 done — skipping
✓ 2017-10 done — skipping
✓ 2017-11 done — skipping
✓ 2017-12 done — skipping
✓ 2018-01 done — skipping
✓ 2018-02 done — skipping
✓ 2018-03 done — skipping
✓ 2018-04 done — skipping
✓ 2018-05 done — skipping
✓ 2018-06 done — skipping
✓ 2018-07 done — skipping
✓ 2018-08 done — skipping
✓ 2018-09 done — skipping
✓ 2018-10 done — skipping
✓ 2018-11 done — skipping
✓ 2018-12 done — skipping
✓ 2019-01 done — skipping
✓ 2019-02 done — skipping
✓ 2019-03 done — skipping
✓ 2019-04 done — skipping
✓ 2019-05 done — skipping
✓ 2019-06 done — skipping
✓ 2019-07 done — skipping
✓ 2019-08 done — skipping
✓ 2019-09 done — skipping
✓ 2019-10 done — skipping
✓ 2019-11 done — skipping
✓ 2019-12 done — skipping
✓ 2020-01 done — skipping
✓ 2020-02 done — skipping
✓ 2020-03 do

In [33]:
import pandas as pd, os
ev = pd.read_parquet("gdelt_chip_events_2017_2023.parquet")
ev["date"] = pd.to_datetime(ev["date"])
print("date range:", ev["date"].min().date(), "→", ev["date"].max().date())
print("\nevents per year:")
print(ev["date"].dt.year.value_counts().sort_index())
print("\nchunk files that got data:")
print(sorted(f for f in os.listdir("gdelt_chunks") if f.endswith(".parquet")))

date range: 1920-01-01 → 2023-12-31

events per year:
date
1920      8437
2007       141
2008       189
2009       222
2010        73
2011        75
2012        72
2013        92
2016      6923
2017    690173
2018    728281
2019    658171
2020    503944
2021    442411
2022    383578
2023    558263
Name: count, dtype: int64

chunk files that got data:
['gdelt_chip_v1_2017-01.parquet', 'gdelt_chip_v1_2017-02.parquet', 'gdelt_chip_v1_2017-03.parquet', 'gdelt_chip_v1_2017-04.parquet', 'gdelt_chip_v1_2017-05.parquet', 'gdelt_chip_v1_2017-06.parquet', 'gdelt_chip_v1_2017-07.parquet', 'gdelt_chip_v1_2017-08.parquet', 'gdelt_chip_v1_2017-09.parquet', 'gdelt_chip_v1_2017-10.parquet', 'gdelt_chip_v1_2017-11.parquet', 'gdelt_chip_v1_2017-12.parquet', 'gdelt_chip_v1_2018-01.parquet', 'gdelt_chip_v1_2018-02.parquet', 'gdelt_chip_v1_2018-03.parquet', 'gdelt_chip_v1_2018-04.parquet', 'gdelt_chip_v1_2018-05.parquet', 'gdelt_chip_v1_2018-06.parquet', 'gdelt_chip_v1_2018-07.parquet', 'gdelt_chip_v1_2018

In [34]:
import time

SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "Mozilla/5.0 (research data pull)"})

def download_gdelt_file_v1(date_str, retries=4, pause=0.4):
    """One GDELT 1.0 daily file, politely — avoids the rate-limit block."""
    url = f"http://data.gdeltproject.org/events/{date_str}.export.CSV.zip"
    for attempt in range(retries):
        try:
            r = SESSION.get(url, timeout=60)
            if r.status_code == 200:
                z = zipfile.ZipFile(io.BytesIO(r.content))
                raw = pd.read_csv(z.open(z.namelist()[0]), sep="\t",
                                  header=None, dtype=str, low_memory=False)
                if raw.shape[1] != len(GDELT_COLUMNS_V1):
                    print(f"  ⚠ {date_str}: {raw.shape[1]} cols (expected {len(GDELT_COLUMNS_V1)})")
                    return None
                raw.columns = GDELT_COLUMNS_V1
                time.sleep(pause)                  # <-- the key fix: don't hammer
                return raw
            if r.status_code == 404:
                return None                        # that day genuinely has no file
            print(f"  {date_str}: HTTP {r.status_code} (throttled?), retry {attempt+1}/{retries}")
            time.sleep(2 ** attempt + 1)           # exponential backoff
        except Exception as e:
            print(f"  {date_str}: {type(e).__name__}, retry {attempt+1}/{retries}")
            time.sleep(2 ** attempt + 1)
    print(f"  {date_str}: gave up after {retries} tries")
    return None

In [35]:
t = download_gdelt_file_v1("20220103")
print(None if t is None else t.shape)   # want something like (180000, 58)

(78231, 58)


In [36]:
events = pull_gdelt_years_v1(2017, 2023)

✓ 2017-01 done — skipping
✓ 2017-02 done — skipping
✓ 2017-03 done — skipping
✓ 2017-04 done — skipping
✓ 2017-05 done — skipping
✓ 2017-06 done — skipping
✓ 2017-07 done — skipping
✓ 2017-08 done — skipping
✓ 2017-09 done — skipping
✓ 2017-10 done — skipping
✓ 2017-11 done — skipping
✓ 2017-12 done — skipping
✓ 2018-01 done — skipping
✓ 2018-02 done — skipping
✓ 2018-03 done — skipping
✓ 2018-04 done — skipping
✓ 2018-05 done — skipping
✓ 2018-06 done — skipping
✓ 2018-07 done — skipping
✓ 2018-08 done — skipping
✓ 2018-09 done — skipping
✓ 2018-10 done — skipping
✓ 2018-11 done — skipping
✓ 2018-12 done — skipping
✓ 2019-01 done — skipping
✓ 2019-02 done — skipping
✓ 2019-03 done — skipping
✓ 2019-04 done — skipping
✓ 2019-05 done — skipping
✓ 2019-06 done — skipping
✓ 2019-07 done — skipping
✓ 2019-08 done — skipping
✓ 2019-09 done — skipping
✓ 2019-10 done — skipping
✓ 2019-11 done — skipping
✓ 2019-12 done — skipping
✓ 2020-01 done — skipping
✓ 2020-02 done — skipping
✓ 2020-03 do

In [37]:
ev = pd.read_parquet("gdelt_chip_events_2017_2023.parquet")
ev["date"] = pd.to_datetime(ev["date"])
ev = ev[ev["date"].dt.year.between(2017, 2023)]   # drops the stray 2007/2016 rows
print(ev["date"].dt.year.value_counts().sort_index())

date
2017    690173
2018    728281
2019    658171
2020    503944
2021    442411
2022    383578
2023    558263
Name: count, dtype: int64


In [38]:
def filter_country_only(df, countries=SEMICONDUCTOR_COUNTRIES):
    """Keep ALL event types — only restrict to the country scope. Filter the rest later."""
    df = df.copy()
    for col in ["NumMentions","NumSources","NumArticles","GoldsteinScale","AvgTone"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df[
        df["Actor1CountryCode"].isin(countries) |
        df["Actor2CountryCode"].isin(countries) |
        df["ActionGeo_CountryCode"].isin(countries)
    ]
    keep_cols = [
        "GLOBALEVENTID","SQLDATE","Actor1Name","Actor1CountryCode",
        "Actor2Name","Actor2CountryCode","ActionGeo_CountryCode","ActionGeo_FullName",
        "EventCode","EventBaseCode","EventRootCode","QuadClass",
        "GoldsteinScale","NumMentions","NumSources","NumArticles","AvgTone","SOURCEURL",
    ]
    return df[[c for c in keep_cols if c in df.columns]]

In [39]:
def filter_country_only(df, countries=SEMICONDUCTOR_COUNTRIES):
    """Keep ALL event types — only restrict to the chip countries. Filter the rest later."""
    df = df.copy()
    for c in ["NumMentions","NumSources","NumArticles","GoldsteinScale","AvgTone"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df[
        df["Actor1CountryCode"].isin(countries) |
        df["Actor2CountryCode"].isin(countries) |
        df["ActionGeo_CountryCode"].isin(countries)
    ]
    keep = ["GLOBALEVENTID","SQLDATE","Actor1Name","Actor1CountryCode",
            "Actor2Name","Actor2CountryCode","ActionGeo_CountryCode","ActionGeo_FullName",
            "EventCode","EventBaseCode","EventRootCode","QuadClass",
            "GoldsteinScale","NumMentions","NumSources","NumArticles","AvgTone","SOURCEURL"]
    return df[[c for c in keep if c in df.columns]]

RAW_DIR = "gdelt_chunks_raw"
os.makedirs(RAW_DIR, exist_ok=True)

def extract_country_raw_v1(start_date, end_date, countries=SEMICONDUCTOR_COUNTRIES,
                           min_mentions=3, output_path="raw_chunk.parquet"):
    parts = []
    for d in tqdm(generate_gdelt_dates(start_date, end_date)):
        df = download_gdelt_file_v1(d)
        if df is None or df.empty:
            continue
        df = filter_country_only(df, countries=countries)   # <-- the "pipe": filter per file
        if min_mentions:
            df = df[df["NumMentions"] >= min_mentions]       # light noise floor
        if not df.empty:
            parts.append(df)
    if not parts:
        print("nothing found"); return pd.DataFrame()
    res = pd.concat(parts, ignore_index=True)
    res["date"] = pd.to_datetime(res["SQLDATE"], format="%Y%m%d", errors="coerce")
    res = res.drop_duplicates("GLOBALEVENTID")
    res.to_parquet(output_path, index=False)
    print(f"saved {len(res):,} -> {output_path}")
    return res

def pull_country_raw_v1(y0=2017, y1=2023, countries=SEMICONDUCTOR_COUNTRIES, min_mentions=3):
    files = []
    for start, end in _month_ranges(y0, y1):
        tag = start[:7]
        out = f"{RAW_DIR}/raw_{tag}.parquet"
        files.append(out)
        if os.path.exists(out):
            print(f"✓ {tag} done"); continue
        print(f"\n=== {tag} (raw country pull) ===")
        extract_country_raw_v1(start, end, countries=countries,
                               min_mentions=min_mentions, output_path=out)
    parts = [pd.read_parquet(f) for f in files if os.path.exists(f)]
    events = pd.concat(parts, ignore_index=True).drop_duplicates("GLOBALEVENTID")
    events.to_parquet("gdelt_country_raw_2017_2023.parquet", index=False)
    print(f"\nFINAL: {len(events):,} -> gdelt_country_raw_2017_2023.parquet")
    return events

# raw_events = pull_country_raw_v1(2017, 2023)

In [ ]:
raw_events = pull_country_raw_v1(2017, 2023)


=== 2017-01 (raw country pull) ===


  3%|▎         | 1/31 [00:02<01:12,  2.40s/it]

In [2]:
import pandas as pd

ev = pd.read_parquet("gdelt_chip_events_2017_2023.parquet")
ev = ev[pd.to_datetime(ev["date"]).dt.year.between(2017, 2023)]
cm = gdelt_to_country_month(ev)
print(sorted(cm["ym"].dt.year.unique()))   # want [2017,...,2023], not just [2024]

NameError: name 'gdelt_to_country_month' is not defined